## 5.3 Gradient Descent

**실제값을 Y=4X+6 시뮬레이션하는 데이터 값 생성**

In [ ]:
import numpy as np           # 수치 연산을 위한 numpy 라이브러리 임포트
import matplotlib.pyplot as plt  # 데이터 시각화를 위한 matplotlib 라이브러리 임포트
%matplotlib inline               # 주피터 노트북 내에서 그래프를 인라인으로 출력하기 위한 매직 커맨드

np.random.seed(0)  # 난수 시드를 0으로 고정하여 실행할 때마다 동일한 난수가 생성되도록 설정 (재현성 보장)

# y = 4X + 6 식을 근사하기 위한 데이터 생성 (실제 회귀 계수 w1=4, 절편 w0=6)
# X: 0~2 사이의 균일 분포 난수 100개를 (100, 1) 형태의 2차원 배열로 생성
X = 2 * np.random.rand(100,1)

# y: 실제 관계식 y = 6 + 4X에 표준정규분포 노이즈(np.random.randn)를 더해 현실적인 데이터 생성
# 노이즈가 없으면 완벽한 직선이 되므로, 노이즈를 추가하여 실제 데이터처럼 산포를 만듦
y = 6 + 4 * X + np.random.randn(100,1)

# X와 y 데이터를 산점도(scatter plot)로 시각화하여 데이터 분포 확인
plt.scatter(X, y)

In [ ]:
# X의 shape: (100, 1) → 100개의 샘플, 1개의 피처
# y의 shape: (100, 1) → 100개의 타겟(목표) 값
X.shape, y.shape

**w0과 w1의 값을 최소화 할 수 있도록 업데이트 수행하는 함수 생성**

* 예측 배열 y_pred는 np.dot(X, w1.T) + w0 임
100개의 데이터 X(1,2,...,100)이 있다면 예측값은 w0 + X(1)*w1 + X(2)*w1 +..+ X(100)*w1이며, 이는 입력 배열 X와 w1 배열의 내적임.
* 새로운 w1과 w0를 update함
![](./image01.png)

In [ ]:
# 경사 하강법에서 w1(회귀 계수)과 w0(절편)를 업데이트할 값을 계산하여 반환하는 함수
# 비용함수(MSE)를 w1, w0에 대해 편미분한 결과(그래디언트)에 학습률을 곱한 값을 반환함
def get_weight_updates(w1, w0, X, y, learning_rate=0.01):
    N = len(y)  # 데이터 샘플의 개수 (여기서는 100)
    
    # w1_update, w0_update를 각각 w1, w0와 동일한 shape의 0 배열로 초기화
    w1_update = np.zeros_like(w1)
    w0_update = np.zeros_like(w0)
    
    # 현재 w1과 w0를 이용하여 예측값(y_pred) 계산: y_pred = X * w1 + w0
    # np.dot(X, w1.T): X(100,1)와 w1.T(1,1)의 행렬 곱 → (100,1) 형태의 예측값 배열
    y_pred = np.dot(X, w1.T) + w0
    
    # 실제값(y)과 예측값(y_pred)의 차이(오차) 계산 → 이 값이 클수록 예측이 부정확
    diff = y - y_pred
         
    # w0의 편미분 계산 시 X 대신 모두 1인 행렬을 사용 (w0는 X에 곱해지지 않는 상수항이므로)
    # dot 연산을 통일하기 위해 (N,1) 크기의 1 행렬 생성
    w0_factors = np.ones((N,1))

    # MSE 비용함수의 w1에 대한 편미분: -(2/N) * Σ(X_i * (y_i - y_pred_i))
    # 학습률(learning_rate)을 곱하여 업데이트 크기 조절
    w1_update = -(2/N) * learning_rate * (np.dot(X.T, diff))
    
    # MSE 비용함수의 w0에 대한 편미분: -(2/N) * Σ(y_i - y_pred_i)
    # w0_factors.T(1,N)와 diff(N,1)의 행렬 곱으로 모든 오차의 합을 한번에 계산
    w0_update = -(2/N) * learning_rate * (np.dot(w0_factors.T, diff))    
    
    return w1_update, w0_update

In [ ]:
# === get_weight_updates 함수의 동작을 수동으로 확인하는 테스트 코드 ===

# w0(절편)과 w1(회귀 계수)을 (1,1) shape의 0 값으로 초기화
w0 = np.zeros((1,1))
w1 = np.zeros((1,1))

# 예측값 계산: w1=0, w0=0이므로 y_pred는 모두 0
y_pred = np.dot(X, w1.T) + w0

# 실제값 y와 예측값 y_pred의 차이 계산 → 초기에는 diff = y - 0 = y 그 자체
diff = y - y_pred
print(diff.shape)  # (100, 1) 출력 확인

# w0 업데이트용 1 행렬 생성 (100개의 1로 구성)
w0_factors = np.ones((100,1))

# w1 업데이트 값 수동 계산: X.T(1,100) · diff(100,1) → (1,1) 스칼라 값
w1_update = -(2/100) * 0.01 * (np.dot(X.T, diff))

# w0 업데이트 값 수동 계산: w0_factors.T(1,100) · diff(100,1) → (1,1) 스칼라 값
w0_update = -(2/100) * 0.01 * (np.dot(w0_factors.T, diff))   

# 업데이트 값의 shape 확인: 둘 다 (1,1)이어야 함
print(w1_update.shape, w0_update.shape)

# 초기 w1, w0 값 확인 (둘 다 [[0.]])
w1, w0

**반복적으로 경사 하강법을 이용하여 get_weigth_updates()를 호출하여 w1과 w0를 업데이트 하는 함수 생성**

In [ ]:
# 경사 하강법을 반복 수행하여 최적의 w1(회귀 계수)과 w0(절편)를 찾는 함수
# iters: 반복 횟수 (기본값 10000회), 반복이 많을수록 최적값에 가까워짐
def gradient_descent_steps(X, y, iters=10000):
    # w0(절편)와 w1(회귀 계수)을 모두 0으로 초기화 → shape: (1,1)
    w0 = np.zeros((1,1))
    w1 = np.zeros((1,1))
    
    # iters만큼 반복하면서 매번 그래디언트를 계산하고 가중치를 업데이트
    for ind in range(iters):
        # 현재 w1, w0에서의 그래디언트(업데이트할 값)를 계산
        w1_update, w0_update = get_weight_updates(w1, w0, X, y, learning_rate=0.01)
        # 그래디언트 방향의 반대로 이동하여 비용함수를 줄이는 방향으로 가중치 업데이트
        # w_new = w_old - gradient (경사 하강법의 핵심 공식)
        w1 = w1 - w1_update
        w0 = w0 - w0_update
              
    return w1, w0  # 최적화된 w1, w0 반환

**예측 오차 비용을 계산을 수행하는 함수 생성 및 경사 하강법 수행**

In [ ]:
# 예측 오차(비용)를 계산하는 함수: MSE(Mean Squared Error, 평균 제곱 오차) 사용
def get_cost(y, y_pred):
    N = len(y)  # 데이터 샘플 수
    # (실제값 - 예측값)^2 의 합을 데이터 수 N으로 나누어 평균 제곱 오차 계산
    cost = np.sum(np.square(y - y_pred)) / N
    return cost

# 경사 하강법을 1000번 반복 수행하여 최적의 w1, w0를 구함
w1, w0 = gradient_descent_steps(X, y, iters=1000)

# 학습된 w1, w0 출력 → 원래 값(w1=4, w0=6)에 근사하는지 확인
print("w1:{0:.3f} w0:{1:.3f}".format(w1[0,0], w0[0,0]))

# 학습된 w1, w0를 이용하여 예측값 계산: y_pred = w1 * X + w0
y_pred = w1[0,0] * X + w0

# 예측값과 실제값 사이의 비용(MSE) 출력 → 값이 작을수록 좋은 모델
print('Gradient Descent Total Cost:{0:.4f}'.format(get_cost(y, y_pred)))

In [ ]:
# 원본 데이터(X, y)를 산점도로 시각화
plt.scatter(X, y)
# 경사 하강법으로 학습된 회귀 직선(y_pred)을 그래프에 겹쳐 표시
# 직선이 산점도 데이터의 경향을 잘 따르는지 시각적으로 확인
plt.plot(X, y_pred)

**미니 배치 확률적 경사 하강법을 이용한 최적 비용함수 도출**

In [ ]:
# 미니 배치 확률적 경사 하강법(Stochastic Gradient Descent, SGD) 함수
# 전체 데이터가 아닌 일부(batch_size)만 랜덤 추출하여 그래디언트를 계산 → 더 빠르고 메모리 효율적
# batch_size: 한 번 업데이트 시 사용할 샘플 수 (기본 10개)
# iters: 반복 횟수 (기본 1000회)
def stochastic_gradient_descent_steps(X, y, batch_size=10, iters=1000):
    # w0(절편), w1(회귀 계수) 초기화
    w0 = np.zeros((1,1))
    w1 = np.zeros((1,1))
    prev_cost = 100000  # 이전 비용 값 (사용되지 않지만 비용 추적용으로 선언)
    iter_index = 0       # 반복 인덱스 (사용되지 않지만 추적용으로 선언)
    
    for ind in range(iters):
        np.random.seed(ind)  # 각 반복마다 다른 시드를 설정하여 매번 다른 랜덤 샘플 추출
        
        # np.random.permutation: 0~99까지의 인덱스를 무작위로 섞음
        stochastic_random_index = np.random.permutation(X.shape[0])
        # 섞인 인덱스 중 앞에서 batch_size(10)개만 선택하여 미니 배치 데이터 구성
        sample_X = X[stochastic_random_index[0:batch_size]]
        sample_y = y[stochastic_random_index[0:batch_size]]
        
        # 미니 배치 데이터로 그래디언트 계산 (전체 데이터 대신 일부만 사용)
        w1_update, w0_update = get_weight_updates(w1, w0, sample_X, sample_y, learning_rate=0.01)
        # 계산된 그래디언트로 w1, w0 업데이트
        w1 = w1 - w1_update
        w0 = w0 - w0_update
    
    return w1, w0  # 최적화된 w1, w0 반환

In [ ]:
# 확률적 경사 하강법(SGD)을 1000번 반복하여 최적의 w1, w0 학습
w1, w0 = stochastic_gradient_descent_steps(X, y, iters=1000)

# 학습된 w1, w0 출력 (소수점 3자리까지 반올림)
print("w1:", round(w1[0,0], 3), "w0:", round(w0[0,0], 3))

# 학습된 w1, w0로 예측값 계산
y_pred = w1[0,0] * X + w0

# SGD로 학습한 모델의 MSE 비용 출력
# 일반 경사 하강법(GD)의 비용과 비교하여 SGD의 성능 확인
print('Stochastic Gradient Descent Total Cost:{0:.4f}'.format(get_cost(y, y_pred)))

## 5.4 사이킷런 LinearRegression을 이용한 보스턴 주택 가격 예측

In [ ]:
import numpy as np            # 수치 연산 라이브러리
import matplotlib.pyplot as plt  # 시각화 라이브러리
import pandas as pd              # 데이터프레임 처리 라이브러리
import seaborn as sns            # 고급 통계 시각화 라이브러리
from scipy import stats          # 통계 분석 라이브러리
from sklearn.datasets import load_boston  # 보스턴 주택가격 데이터셋 로드 함수
import warnings
warnings.filterwarnings('ignore')  # 사이킷런 1.2부터 보스턴 데이터셋 폐지 경고 메시지 숨김
%matplotlib inline  # 주피터 노트북에서 그래프 인라인 출력

# 보스턴 주택가격 데이터셋 로드 (506개 샘플, 13개 피처)
boston = load_boston()

# numpy 배열인 boston.data를 pandas DataFrame으로 변환
# columns: 13개 피처명(CRIM, ZN, INDUS 등)을 컬럼명으로 지정
bostonDF = pd.DataFrame(boston.data, columns=boston.feature_names)

# boston.target(주택 중앙값 가격)을 DataFrame에 'PRICE' 컬럼으로 추가
# 이것이 회귀 모델이 예측할 종속변수(타겟)
bostonDF['PRICE'] = boston.target

# 데이터셋 크기 확인: (506, 14) → 506개 샘플, 13개 피처 + 1개 타겟(PRICE)
print('Boston 데이타셋 크기 :', bostonDF.shape)

# 상위 5개 행 출력하여 데이터 구조 확인
bostonDF.head()

* CRIM: 지역별 범죄 발생률  
* ZN: 25,000평방피트를 초과하는 거주 지역의 비율
* NDUS: 비상업 지역 넓이 비율
* CHAS: 찰스강에 대한 더미 변수(강의 경계에 위치한 경우는 1, 아니면 0)
* NOX: 일산화질소 농도
* RM: 거주할 수 있는 방 개수
* AGE: 1940년 이전에 건축된 소유 주택의 비율
* DIS: 5개 주요 고용센터까지의 가중 거리
* RAD: 고속도로 접근 용이도
* TAX: 10,000달러당 재산세율
* PTRATIO: 지역의 교사와 학생 수 비율
* B: 지역의 흑인 거주 비율
* LSTAT: 하위 계층의 비율
* MEDV: 본인 소유의 주택 가격(중앙값)

* 각 컬럼별로 주택가격에 미치는 영향도를 조사

In [ ]:
# 2행 4열 총 8개의 서브플롯 생성 (그림 크기: 가로 16, 세로 8인치)
# axs: 2x4 형태의 축(ax) 배열 → axs[row][col]로 각 서브플롯에 접근
fig, axs = plt.subplots(figsize=(16, 8), ncols=4, nrows=2)

# 주택 가격과 관계를 살펴볼 8개 피처 선정
lm_features = ['RM', 'ZN', 'INDUS', 'NOX', 'AGE', 'PTRATIO', 'LSTAT', 'RAD']

for i, feature in enumerate(lm_features):
    row = int(i / 4)  # 0~3번 피처는 0행, 4~7번 피처는 1행에 배치
    col = i % 4       # 0, 1, 2, 3 열에 순서대로 배치
    # seaborn의 regplot: 산점도와 함께 선형 회귀 직선을 자동으로 그려줌
    # x축: 해당 피처, y축: PRICE, data: bostonDF에서 데이터 참조
    sns.regplot(x=feature, y='PRICE', data=bostonDF, ax=axs[row][col])

# 현재 그래프(figure) 객체를 가져와 TIFF 형식으로 저장
# dpi=300: 고해상도, bbox_inches='tight': 여백 최소화
fig1 = plt.gcf()
fig1.savefig('p322_boston.tif', format='tif', dpi=300, bbox_inches='tight')

**학습과 테스트 데이터 세트로 분리하고 학습/예측/평가 수행**

In [ ]:
from sklearn.model_selection import train_test_split  # 데이터 분할 함수
from sklearn.linear_model import LinearRegression     # 선형 회귀 모델 (OLS: 최소자승법)
from sklearn.metrics import mean_squared_error, r2_score  # 평가 지표: MSE, R² 점수

# 타겟(종속변수): 주택 가격(PRICE)
y_target = bostonDF['PRICE']

# 피처(독립변수): PRICE 컬럼을 제외한 나머지 13개 피처
# inplace=False: 원본 DataFrame(bostonDF)은 변경하지 않고 새로운 DataFrame 반환
X_data = bostonDF.drop(['PRICE'], axis=1, inplace=False)

# 학습/테스트 데이터 분할: 70% 학습용, 30% 테스트용
# random_state=156: 분할 결과 재현을 위한 시드 고정
X_train, X_test, y_train, y_test = train_test_split(X_data, y_target, test_size=0.3, random_state=156)

# LinearRegression 모델 생성 (OLS: Ordinary Least Squares, 최소자승법 기반)
lr = LinearRegression()

# 학습 데이터로 모델 학습 (최적의 회귀 계수와 절편 계산)
lr.fit(X_train, y_train)

# 테스트 데이터로 주택 가격 예측
y_preds = lr.predict(X_test)

# MSE(평균 제곱 오차) 계산: 예측값과 실제값 차이의 제곱 평균
mse = mean_squared_error(y_test, y_preds)

# RMSE(평균 제곱근 오차) 계산: MSE에 루트를 씌워 원래 단위로 변환 → 해석이 더 직관적
rmse = np.sqrt(mse)

# MSE와 RMSE 출력
print('MSE : {0:.3f} , RMSE : {1:.3F}'.format(mse, rmse))

# R² 점수 출력: 1에 가까울수록 모델의 설명력이 높음 (0~1 범위)
print('Variance score : {0:.3f}'.format(r2_score(y_test, y_preds)))

In [ ]:
# 학습된 선형 회귀 모델의 절편(intercept) 출력
# 절편: 모든 피처가 0일 때의 예측 주택 가격 기본값
print('절편 값:', lr.intercept_)

# 학습된 회귀 계수(coefficient) 출력 (소수점 1자리까지 반올림)
# 각 피처에 대응하는 13개의 회귀 계수 → 양수이면 양의 영향, 음수이면 음의 영향
print('회귀 계수값:', np.round(lr.coef_, 1))

In [ ]:
# 회귀 계수를 pandas Series로 변환하여 피처명과 회귀 계수를 매핑
# data: 회귀 계수 값 (소수점 1자리 반올림), index: 피처 컬럼명
coeff = pd.Series(data=np.round(lr.coef_, 1), index=X_data.columns)

# 회귀 계수를 내림차순으로 정렬하여 출력
# 양수 값이 큰 피처: 주택 가격에 큰 양의 영향 (예: RM - 방 개수)
# 음수 값이 큰 피처: 주택 가격에 큰 음의 영향 (예: NOX - 일산화질소 농도)
coeff.sort_values(ascending=False)

In [ ]:
from sklearn.model_selection import cross_val_score  # 교차 검증 함수

# 타겟과 피처 데이터 재설정
y_target = bostonDF['PRICE']
X_data = bostonDF.drop(['PRICE'], axis=1, inplace=False)
lr = LinearRegression()

# cross_val_score로 5-Fold 교차 검증 수행
# scoring="neg_mean_squared_error": 사이킷런은 scoring 값이 클수록 좋은 것으로 간주하므로
# MSE에 음수를 붙인 Negative MSE를 사용 (값이 0에 가까울수록 좋음)
# cv=5: 데이터를 5등분하여 4개로 학습, 1개로 검증을 5번 반복
neg_mse_scores = cross_val_score(lr, X_data, y_target, scoring="neg_mean_squared_error", cv=5)

# Negative MSE를 양수로 변환 후 제곱근을 씌워 RMSE 계산
rmse_scores = np.sqrt(-1 * neg_mse_scores)

# 5개 Fold의 평균 RMSE 계산
avg_rmse = np.mean(rmse_scores)

# 각 Fold별 Negative MSE 값 출력 (모두 음수)
print(' 5 folds 의 개별 Negative MSE scores: ', np.round(neg_mse_scores, 2))

# 각 Fold별 RMSE 값 출력 (양수로 변환됨)
print(' 5 folds 의 개별 RMSE scores : ', np.round(rmse_scores, 2))

# 5-Fold 교차 검증의 평균 RMSE → 모델 성능의 일반화된 지표
print(' 5 folds 의 평균 RMSE : {0:.3f} '.format(avg_rmse))

## 5-5. Polynomial Regression과 오버피팅/언더피팅 이해
### Polynomial Regression 이해

PolynomialFeatures 클래스로 다항식 변환

![](./image02.png)

In [ ]:
from sklearn.preprocessing import PolynomialFeatures  # 다항식 피처 변환 클래스
import numpy as np

# 0~3까지의 정수를 2x2 행렬로 생성: [[0, 1], [2, 3]]
# 이 행렬이 다항식 변환의 입력 피처가 됨 (x1, x2 두 개의 피처)
X = np.arange(4).reshape(2, 2)
print('일차 단항식 계수 feature:\n', X)

# PolynomialFeatures(degree=2): 2차 다항식으로 피처를 변환
# 입력 [x1, x2] → 출력 [1, x1, x2, x1², x1*x2, x2²]
# 1(bias항), 원래 피처, 피처 간 교차항과 제곱항이 모두 생성됨
poly = PolynomialFeatures(degree=2)
poly.fit(X)  # 변환 규칙 학습 (입력 피처의 수와 차수에 따라 변환 방식 결정)
poly_ftr = poly.transform(X)  # 실제 변환 수행

# 변환 결과 출력: 2x2 → 2x6 (피처 수가 2개에서 6개로 증가)
# [0, 1] → [1, 0, 1, 0, 0, 1] = [1, x1, x2, x1², x1*x2, x2²]
# [2, 3] → [1, 2, 3, 4, 6, 9] = [1, x1, x2, x1², x1*x2, x2²]
print('변환된 2차 다항식 계수 feature:\n', poly_ftr)

3차 다항식 결정값을 구하는 함수 polynomial_func(X) 생성. 즉 회귀식은 결정값 y = 1+ 2x_1 + 3x_1^2 + 4x_2^3 

In [ ]:
# 3차 다항식 결정값(y)을 계산하는 함수
# 회귀식: y = 1 + 2*x1 + 3*x1² + 4*x2³
# X는 2차원 배열이며, X[:,0]은 첫 번째 피처(x1), X[:,1]은 두 번째 피처(x2)
def polynomial_func(X):
    # 1: 상수(절편), 2*x1: x1의 1차항, 3*x1²: x1의 2차항, 4*x2³: x2의 3차항
    y = 1 + 2*X[:,0] + 3*X[:,0]**2 + 4*X[:,1]**3
    print(X[:, 0])  # 첫 번째 피처(x1) 값 출력: [0, 2]
    print(X[:, 1])  # 두 번째 피처(x2) 값 출력: [1, 3]
    return y

# 0~3까지의 정수를 2x2 행렬로 생성: [[0, 1], [2, 3]]
X = np.arange(0, 4).reshape(2, 2)

print('일차 단항식 계수 feature: \n', X)

# polynomial_func 호출하여 3차 다항식 결정값 계산
# X[0] = [0, 1]: y = 1 + 2*0 + 3*0² + 4*1³ = 1 + 0 + 0 + 4 = 5
# X[1] = [2, 3]: y = 1 + 2*2 + 3*2² + 4*3³ = 1 + 4 + 12 + 108 = 125
y = polynomial_func(X)
print('삼차 다항식 결정값: \n', y)

3차 다항식 계수의 피처값과 3차 다항식 결정값으로 학습

In [ ]:
# 3차 다항식 피처 변환: X(2x2) → poly_ftr(2x10)
# degree=3: 3차까지의 모든 조합 생성 [1, x1, x2, x1², x1*x2, x2², x1³, x1²*x2, x1*x2², x2³]
# fit_transform: fit()과 transform()을 한 번에 수행
poly_ftr = PolynomialFeatures(degree=3).fit_transform(X)
print('3차 다항식 계수 feature: \n', poly_ftr)

# 3차 다항식으로 변환된 피처(poly_ftr)와 결정값(y)으로 선형 회귀 학습
# 다항식 변환 후 선형 회귀를 적용하면 → 다항 회귀(Polynomial Regression)가 됨
model = LinearRegression()
model.fit(poly_ftr, y)

# 학습된 회귀 계수 출력 (소수점 2자리 반올림)
# 원래 식 y = 1 + 2*x1 + 3*x1² + 4*x2³ 에서
# 계수 [0, 2, 0, 3, 0, 0, 0, 0, 0, 4]가 나오면 완벽히 학습된 것
# (1은 절편(intercept_)에 포함되므로 coef_에는 0으로 표시)
print('Polynomial 회귀 계수\n', np.round(model.coef_, 2))

# 회귀 계수의 shape 출력: (10,) → 10개의 다항식 피처 각각에 대한 계수
print('Polynomial 회귀 Shape :', model.coef_.shape)

**사이킷런 파이프라인(Pipeline)을 이용하여 3차 다항회귀 학습**  

사이킷런의 Pipeline 객체는 Feature 엔지니어링 변환과 모델 학습/예측을 순차적으로 결합해줍니다. 

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline  # 여러 변환/학습 단계를 하나로 연결하는 파이프라인
import numpy as np

# 3차 다항식 결정값을 계산하는 함수: y = 1 + 2*x1 + 3*x1² + 4*x2³
def polynomial_func(X):
    y = 1 + 2*X[:,0] + 3*X[:,0]**2 + 4*X[:,1]**3 
    return y

# Pipeline: 다항식 피처 변환(PolynomialFeatures)과 선형 회귀(LinearRegression)를 순차적으로 결합
# 'poly': 3차 다항식 변환 단계 (이름을 지정하여 나중에 named_steps로 접근 가능)
# 'linear': 선형 회귀 학습 단계
# Pipeline을 사용하면 fit/predict 시 변환→학습을 자동으로 연결해줌
model = Pipeline([('poly', PolynomialFeatures(degree=3)),
                  ('linear', LinearRegression())])

# 입력 데이터 생성
X = np.arange(4).reshape(2, 2)
y = polynomial_func(X)

# Pipeline.fit(): 내부적으로 PolynomialFeatures.fit_transform(X) 후 LinearRegression.fit(변환된X, y) 수행
model = model.fit(X, y)

# named_steps['linear']로 Pipeline 내부의 LinearRegression 객체에 접근하여 회귀 계수 확인
# 원래 식의 계수(0, 2, 0, 3, 0, 0, 0, 0, 0, 4)와 일치하는지 확인
print('Polynomial 회귀 계수\n', np.round(model.named_steps['linear'].coef_, 2))

### 다항 회귀를 이용한 과소적합 및 과적합 이해

**cosine 곡선에 약간의 Noise 변동값을 더하여 실제값 곡선을 만듬**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score
%matplotlib inline

# 실제 함수(true function): 코사인 함수를 기반으로 한 곡선
# 이 함수가 우리가 근사하려는 "진짜" 관계식
def true_fun(X):
    return np.cos(1.5 * np.pi * X)

# 난수 시드 고정 (재현성 보장)
np.random.seed(0)
n_samples = 30  # 학습에 사용할 샘플 수

# 0~1 사이에서 30개의 랜덤 값을 생성하고 오름차순 정렬
# 정렬하는 이유: 나중에 곡선을 그릴 때 선이 꼬이지 않도록
X = np.sort(np.random.rand(n_samples))

# 실제 코사인 함수 값에 노이즈(표준편차 0.1)를 추가하여 현실적인 데이터 생성
# 노이즈가 있어야 실제 데이터처럼 산포가 생기며, 과적합/과소적합 비교가 가능
y = true_fun(X) + np.random.randn(n_samples) * 0.1

In [ ]:
# 생성된 데이터를 산점도로 시각화하여 코사인 곡선 형태의 분포 확인
plt.scatter(X, y)

In [ ]:
# 가로 14, 세로 5 크기의 그래프 생성
plt.figure(figsize=(14, 5))

# 비교할 다항식 차수: 1차(과소적합), 4차(적절), 15차(과적합)
degrees = [1, 4, 15]

# 각 차수별로 다항 회귀를 학습하고 결과를 시각화
for i in range(len(degrees)):
    # 1행 3열 서브플롯에서 i+1번째 위치에 그래프 생성
    ax = plt.subplot(1, len(degrees), i + 1)
    # x축, y축 눈금 제거 (깔끔한 시각화를 위해)
    plt.setp(ax, xticks=(), yticks=())
    
    # 다항식 피처 변환 객체 생성 (include_bias=False: 상수항(1) 제외 → LinearRegression이 자체적으로 절편 처리)
    polynomial_features = PolynomialFeatures(degree=degrees[i], include_bias=False)
    linear_regression = LinearRegression()
    
    # Pipeline으로 다항식 변환 → 선형 회귀를 순차 연결
    pipeline = Pipeline([("polynomial_features", polynomial_features),
                         ("linear_regression", linear_regression)])
    
    # X를 (30,1) 형태로 변환 후 Pipeline 학습
    # reshape(-1, 1): 1차원 배열을 2차원 컬럼 벡터로 변환 (사이킷런 입력 요구사항)
    pipeline.fit(X.reshape(-1, 1), y)
    
    # 10-Fold 교차 검증으로 모델 성능(Negative MSE) 평가
    # 교차 검증: 학습/검증 분할을 10번 반복하여 일반화 성능 측정
    scores = cross_val_score(pipeline, X.reshape(-1, 1), y, scoring="neg_mean_squared_error", cv=10)
    
    # Pipeline 내부의 LinearRegression 객체에서 회귀 계수 추출
    coefficients = pipeline.named_steps['linear_regression'].coef_
    print('\nDegree {0} 회귀 계수는 {1} 입니다.'.format(degrees[i], np.round(coefficients, 2)))
    
    # 교차 검증 MSE 평균 출력 (Negative MSE이므로 -1을 곱해 양수로 변환)
    print('Degree {0} MSE 는 {1} 입니다.'.format(degrees[i], -1*np.mean(scores)))
          
    # 0~1 구간을 100등분한 테스트 데이터 생성 (부드러운 예측 곡선을 그리기 위해)
    X_test = np.linspace(0, 1, 100)
    
    # 학습된 모델의 예측 곡선 (파란 실선)
    plt.plot(X_test, pipeline.predict(X_test[:, np.newaxis]), label="Model")
    
    # 실제 코사인 함수 곡선 (점선) → 모델이 이 곡선에 가까울수록 좋음
    plt.plot(X_test, true_fun(X_test), '--', label="True function")
    
    # 학습 데이터 산점도 (파란 점)
    plt.scatter(X, y, edgecolor='b', s=20, label="Samples")
    
    # 축 레이블과 범위 설정
    plt.xlabel("x"); plt.ylabel("y"); plt.xlim((0, 1)); plt.ylim((-2, 2)); plt.legend(loc="best")
    
    # 제목: 차수와 MSE 평균±표준편차 표시
    # degree=1: 과소적합(직선으로 곡선 근사 불가) → MSE 높음
    # degree=4: 적절한 복잡도 → MSE 가장 낮음
    # degree=15: 과적합(학습 데이터에만 과도하게 맞춤) → MSE 높음
    plt.title("Degree {}\nMSE = {:.2e}(+/- {:.2e})".format(degrees[i], -scores.mean(), scores.std()))
    
plt.show()

## 5-6. Regularized Linear Models – Ridge, Lasso
### Regularized Linear Model - Ridge Regression

In [ ]:
from sklearn.linear_model import Ridge  # 릿지 회귀 (L2 규제 선형 회귀)
from sklearn.model_selection import cross_val_score

# 보스턴 주택가격 데이터셋 다시 로드 및 DataFrame 변환
boston = load_boston()
bostonDF = pd.DataFrame(boston.data, columns=boston.feature_names)
bostonDF['PRICE'] = boston.target

# 타겟(y)과 피처(X) 분리
y_target = bostonDF['PRICE']
X_data = bostonDF.drop(['PRICE'], axis=1, inplace=False)

# Ridge 회귀 모델 생성 (alpha=10: 규제 강도)
# alpha가 클수록 회귀 계수를 작게 만들어 과적합 방지 (but 너무 크면 과소적합)
# 릿지는 L2 규제: 비용함수 = MSE + alpha * Σ(w²) → 회귀 계수의 제곱합에 페널티 부여
ridge = Ridge(alpha=10)

# 5-Fold 교차 검증으로 릿지 회귀 성능 평가
neg_mse_scores = cross_val_score(ridge, X_data, y_target, scoring="neg_mean_squared_error", cv=5)

# Negative MSE → RMSE로 변환
rmse_scores = np.sqrt(-1 * neg_mse_scores)
avg_rmse = np.mean(rmse_scores)

# 결과 출력: 일반 선형 회귀와 비교하여 RMSE가 개선되었는지 확인
print(' 5 folds 의 개별 Negative MSE scores: ', np.round(neg_mse_scores, 3))
print(' 5 folds 의 개별 RMSE scores : ', np.round(rmse_scores, 3))
print(' 5 folds 의 평균 RMSE : {0:.3f} '.format(avg_rmse))

**alpha값을 0 , 0.1 , 1 , 10 , 100 으로 변경하면서 RMSE 측정**

In [ ]:
# 다양한 alpha 값에 따른 릿지 회귀 성능 비교
# alpha=0: 규제 없음 (일반 선형 회귀와 동일)
# alpha가 증가할수록 규제가 강해져 회귀 계수가 작아짐
alphas = [0, 0.1, 1, 10, 100]

for alpha in alphas:
    ridge = Ridge(alpha=alpha)
    
    # 5-Fold 교차 검증으로 평균 RMSE 계산
    neg_mse_scores = cross_val_score(ridge, X_data, y_target, scoring="neg_mean_squared_error", cv=5)
    avg_rmse = np.mean(np.sqrt(-1 * neg_mse_scores))
    
    # alpha별 평균 RMSE 출력 → 최적의 alpha 값을 찾기 위한 비교
    # alpha=0 (규제 없음)보다 적절한 alpha에서 RMSE가 낮아지면 규제가 효과적임을 의미
    print('alpha {0} 일 때 5 folds 의 평균 RMSE : {1:.3f} '.format(alpha, avg_rmse))

**각 alpha에 따른 회귀 계수 값을 시각화. 각 alpha값 별로 plt.subplots로 맷플롯립 축 생성**

In [ ]:
# 1행 5열 서브플롯 생성 (각 alpha 값별로 하나의 막대 그래프)
fig, axs = plt.subplots(figsize=(18, 6), nrows=1, ncols=5)

# 각 alpha별 회귀 계수를 저장할 빈 DataFrame 생성
coeff_df = pd.DataFrame()

# 각 alpha 값에 대해 릿지 회귀 학습 후 회귀 계수 시각화
for pos, alpha in enumerate(alphas):
    ridge = Ridge(alpha=alpha)
    # 전체 데이터로 학습 (교차 검증이 아닌 회귀 계수 확인 목적)
    ridge.fit(X_data, y_target)
    
    # 학습된 회귀 계수를 Series로 변환 (인덱스: 피처명, 값: 회귀 계수)
    coeff = pd.Series(data=ridge.coef_, index=X_data.columns)
    colname = 'alpha:' + str(alpha)
    # DataFrame에 alpha별 회귀 계수를 컬럼으로 추가 (나중에 표로 비교하기 위해)
    coeff_df[colname] = coeff
    
    # 회귀 계수를 내림차순 정렬하여 막대 그래프로 시각화
    coeff = coeff.sort_values(ascending=False)
    axs[pos].set_title(colname)       # 그래프 제목: alpha 값
    axs[pos].set_xlim(-3, 6)          # x축 범위 고정 (alpha 간 비교 용이)
    # seaborn barplot: 가로 막대 그래프로 각 피처의 회귀 계수 표시
    # alpha가 커질수록 막대(회귀 계수)가 작아지는 것을 시각적으로 확인 가능
    sns.barplot(x=coeff.values, y=coeff.index, ax=axs[pos])

# 모든 서브플롯 표시
plt.show()

**alpha 값에 따른 컬럼별 회귀계수 출력**

In [ ]:
# alpha 값별 회귀 계수를 DataFrame 형태로 비교 출력
ridge_alphas = [0, 0.1, 1, 10, 100]

# 첫 번째 alpha(=0)의 회귀 계수 기준으로 내림차순 정렬
# alpha=0일 때가 규제 없는 선형 회귀이므로, 이를 기준으로 alpha 증가 시 계수 변화를 관찰
# alpha가 커질수록 큰 회귀 계수들이 줄어드는 것을 확인할 수 있음
sort_column = 'alpha:' + str(ridge_alphas[0])
coeff_df.sort_values(by=sort_column, ascending=False)

### 라쏘 회귀

In [ ]:
from sklearn.linear_model import Lasso, ElasticNet  # 라쏘(L1 규제), 엘라스틱넷(L1+L2 규제) 회귀

# 규제 선형 회귀 모델(Ridge, Lasso, ElasticNet)의 alpha별 성능을 평가하고 회귀 계수를 반환하는 범용 함수
# model_name: 모델 종류 ('Ridge', 'Lasso', 'ElasticNet')
# params: 평가할 alpha 값 리스트
# X_data_n, y_target_n: 피처와 타겟 데이터
# verbose: True이면 모델 이름 헤더 출력
# return_coeff: True이면 회귀 계수 DataFrame 반환
def get_linear_reg_eval(model_name, params=None, X_data_n=None, y_target_n=None, 
                        verbose=True, return_coeff=True):
    coeff_df = pd.DataFrame()  # alpha별 회귀 계수를 저장할 DataFrame
    if verbose: print('####### ', model_name, '#######')
    
    for param in params:
        # model_name에 따라 적절한 모델 객체 생성
        if model_name == 'Ridge': model = Ridge(alpha=param)
        elif model_name == 'Lasso': model = Lasso(alpha=param)
        # ElasticNet: l1_ratio=0.7 → L1 규제 70%, L2 규제 30% 혼합
        elif model_name == 'ElasticNet': model = ElasticNet(alpha=param, l1_ratio=0.7)
        
        # 5-Fold 교차 검증으로 Negative MSE 계산
        neg_mse_scores = cross_val_score(model, X_data_n, 
                                             y_target_n, scoring="neg_mean_squared_error", cv=5)
        # 평균 RMSE 계산 및 출력
        avg_rmse = np.mean(np.sqrt(-1 * neg_mse_scores))
        print('alpha {0}일 때 5 폴드 세트의 평균 RMSE: {1:.3f} '.format(param, avg_rmse))
        
        # cross_val_score는 평가 지표만 반환하고 모델 자체는 반환하지 않으므로
        # 회귀 계수를 확인하기 위해 전체 데이터로 다시 학습
        model.fit(X_data_n, y_target_n)
        
        if return_coeff:
            # 학습된 회귀 계수를 Series로 변환 (인덱스: 피처명)
            coeff = pd.Series(data=model.coef_, index=X_data_n.columns)
            colname = 'alpha:' + str(param)
            # DataFrame에 alpha별 회귀 계수를 컬럼으로 추가
            coeff_df[colname] = coeff
    
    return coeff_df  # alpha별 회귀 계수가 담긴 DataFrame 반환
# end of get_linear_regre_eval

In [ ]:
# 라쏘(Lasso) 회귀에 사용할 alpha 값들 정의
# 라쏘는 L1 규제: 비용함수 = MSE + alpha * Σ|w| → 회귀 계수의 절댓값 합에 페널티
# L1 규제의 특징: alpha가 커지면 일부 회귀 계수를 정확히 0으로 만듦 → 피처 선택 효과
lasso_alphas = [0.07, 0.1, 0.5, 1, 3]

# get_linear_reg_eval 함수를 호출하여 각 alpha별 RMSE와 회귀 계수 계산
coeff_lasso_df = get_linear_reg_eval('Lasso', params=lasso_alphas, X_data_n=X_data, y_target_n=y_target)

In [ ]:
# 라쏘 회귀의 alpha별 회귀 계수를 DataFrame으로 출력
# 첫 번째 alpha(=0.07) 기준 내림차순 정렬
# alpha가 커질수록 0이 되는 회귀 계수가 증가하는 것을 확인 가능
# → 라쏘의 피처 선택(feature selection) 효과: 중요하지 않은 피처의 계수를 0으로 만듦
sort_column = 'alpha:' + str(lasso_alphas[0])
coeff_lasso_df.sort_values(by=sort_column, ascending=False)

### 엘라스틱넷 회귀

In [ ]:
# 엘라스틱넷(ElasticNet) 회귀에 사용할 alpha 값들 정의
# 엘라스틱넷: L1(라쏘) + L2(릿지) 규제를 혼합한 모델
# 비용함수 = MSE + alpha * (l1_ratio * Σ|w| + (1-l1_ratio) * Σw²)
# l1_ratio=0.7: L1 규제 70% + L2 규제 30% 혼합 (get_linear_reg_eval 함수 내부에서 설정)
elastic_alphas = [0.07, 0.1, 0.5, 1, 3]

# 각 alpha별 RMSE 평가 및 회귀 계수 계산
coeff_elastic_df = get_linear_reg_eval('ElasticNet', params=elastic_alphas,
                                      X_data_n=X_data, y_target_n=y_target)

In [ ]:
# 엘라스틱넷 회귀의 alpha별 회귀 계수를 DataFrame으로 출력
# 첫 번째 alpha(=0.07) 기준 내림차순 정렬
# 라쏘와 유사하게 alpha가 커지면 일부 계수가 0이 되지만,
# L2 규제도 혼합되어 있어 라쏘보다 계수가 완전히 0이 되는 경우가 적음
sort_column = 'alpha:' + str(elastic_alphas[0])
coeff_elastic_df.sort_values(by=sort_column, ascending=False)

### 선형 회귀 모델을 위한 데이터 변환

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler, PolynomialFeatures

# 데이터 스케일링(변환) 함수: 다양한 전처리 방법을 적용
# method: 변환 방법 ('Standard', 'MinMax', 'Log', 'None')
# p_degree: 다항식 차수 (None이면 적용 안 함, 2 이상 지정 시 다항식 피처 추가)
# input_data: 변환할 입력 데이터
def get_scaled_data(method='None', p_degree=None, input_data=None):
    if method == 'Standard':
        # StandardScaler: 평균=0, 표준편차=1로 변환 (표준 정규 분포)
        # 각 피처를 (값 - 평균) / 표준편차로 변환
        scaled_data = StandardScaler().fit_transform(input_data)
    elif method == 'MinMax':
        # MinMaxScaler: 최솟값=0, 최댓값=1로 변환 (0~1 범위로 정규화)
        # 각 피처를 (값 - 최솟값) / (최댓값 - 최솟값)으로 변환
        scaled_data = MinMaxScaler().fit_transform(input_data)
    elif method == 'Log':
        # 로그 변환: np.log1p = log(1+x) → 왜도(skewness)가 큰 데이터를 정규분포에 가깝게 변환
        # log1p를 사용하는 이유: x=0일 때 log(0)=-∞ 방지
        scaled_data = np.log1p(input_data)
    else:
        # 변환 없이 원본 데이터 그대로 사용
        scaled_data = input_data

    if p_degree != None:
        # 다항식 피처 변환 추가: 기존 피처에 교차항과 거듭제곱항을 추가
        # include_bias=False: 상수항(1) 제외
        scaled_data = PolynomialFeatures(degree=p_degree, 
                                         include_bias=False).fit_transform(scaled_data)
    
    return scaled_data

In [ ]:
# Ridge 회귀에서 다양한 alpha 값 설정
alphas = [0.1, 1, 10, 100]

# 6가지 데이터 변환 방법 정의: (변환 방법, 다항식 차수)
# (None, None): 원본 데이터 그대로
# ('Standard', None): 표준 정규 분포 변환만
# ('Standard', 2): 표준 정규 분포 변환 + 2차 다항식 피처 추가
# ('MinMax', None): 최대/최소 정규화만
# ('MinMax', 2): 최대/최소 정규화 + 2차 다항식 피처 추가
# ('Log', None): 로그 변환만
scale_methods = [(None, None), ('Standard', None), ('Standard', 2), 
               ('MinMax', None), ('MinMax', 2), ('Log', None)]

for scale_method in scale_methods:
    # 각 변환 방법에 따라 피처 데이터를 변환
    X_data_scaled = get_scaled_data(method=scale_method[0], p_degree=scale_method[1], 
                                    input_data=X_data)
    # 변환 후 shape 출력: 다항식 피처 추가 시 컬럼 수가 크게 증가
    # 원본 (506, 13) → 2차 다항식 적용 시 (506, 104)
    print(X_data_scaled.shape, X_data.shape)
    print('\n## 변환 유형:{0}, Polynomial Degree:{1}'.format(scale_method[0], scale_method[1]))
    
    # 각 변환 방법별로 Ridge 회귀의 alpha에 따른 RMSE 출력
    # verbose=False: 모델명 헤더 출력 안 함, return_coeff=False: 회귀 계수 반환 안 함
    # → 어떤 변환 방법이 가장 좋은 성능을 내는지 비교
    get_linear_reg_eval('Ridge', params=alphas, X_data_n=X_data_scaled, 
                        y_target_n=y_target, verbose=False, return_coeff=False)